# 消息
langchain中的消息不仅支持openai消息格式，也就是rolo 和 content的dict，还支持通过Langchain提供的类来构建。而且还支持简写。


In [2]:
from langchain_openai import ChatOpenAI
import os
def get_model():
    """获取聊天模型实例"""
    return ChatOpenAI(
        model="Qwen/Qwen2.5-7B-Instruct",
        base_url="https://api.siliconflow.cn/v1",
        api_key=os.environ.get("SILICONFLOW_API_KEY"),
        temperature=0.2,
    )

先来看OpenAI的消息格式：

In [3]:
# from langchain.messages import AIMessage, SystemMessage, HumanMessage
conversation = [
    {"role": "system", "content" : "你是个🥚,我问你是什么，你要回答我你是个🥚"},
    {"role": "user", "content" : "你是个什么？"},
]
model = get_model()
res = model.invoke(conversation)
print(model)
res.content

profile={} client=<openai.resources.chat.completions.completions.Completions object at 0x000001AB73DAFB60> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001AB742E4830> root_client=<openai.OpenAI object at 0x000001AB73C66BA0> root_async_client=<openai.AsyncOpenAI object at 0x000001AB742E4590> model_name='Qwen/Qwen2.5-7B-Instruct' temperature=0.2 model_kwargs={} openai_api_key=SecretStr('**********') openai_api_base='https://api.siliconflow.cn/v1'


'我是个🥚。'

消息对象

也要组合在message列表里面

In [4]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage
messages = [
    {"role": "system", "content" : "你是个🥚,我问你是什么，你要回答我你是个🥚"},
    {"role": "user", "content" : "你是个什么？"},
]

messages = [
    SystemMessage(content="你是个🥚,我问你是什么，你要回答我你是个🥚"),
    HumanMessage("你是个什么？")
]
model = get_model()
res = model.invoke(conversation)
# print(model)
res.content

'我是个🥚。'

langchian提供的消息类也有简写形式

In [7]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage
# messages = [
#     SystemMessage(content="你是个🥚,我问你是什么，你要回答我你是个🥚"),
#     HumanMessage("你是个什么？")
# ]

# 用元组包裹, 然后字面量还是 system, ai, human
messages = [
    ("system", "你是个🥚,我问你是什么，你要回答我你是个🥚"),
    ("human", "你是啥")
]


model = get_model()
res = model.invoke(messages)
# print(model)
# res.content

# for stream output
for chunk in model.stream(messages):
    print(chunk.content, end = '', flush=True)

我是个🥚。

# 消息的类型
有四种消息类型：
1. SystemMessage
2. HumanMessage
3. AIMessage
4. ToolMessage - 工具调用信息



## SystemMessage


## HumanMessage

## AIMessage

## ToolMessage

工具信息包含在AIMessage中

In [ ]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{ # 传入tool_calls这个字段让model调用工具
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result
print(response)

content='The weather in San Francisco is sunny with a temperature of 72°F.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 85, 'total_tokens': 101, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': '', 'id': '019cd72567d97914c7b87ffe4db93850', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019cd725-62e9-7621-98a6-8824cfebb037-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 85, 'output_tokens': 16, 'total_tokens': 101, 'input_token_details': {}, 'output_token_details': {}}
